<a href="https://colab.research.google.com/github/PlaZMaD/GP_ML_2026/blob/main/Seminar_2_HW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание к семинару 2

Задание проверяет темы семинара 2: регуляризацию, метрики классификации, деревья и ансамбли, кросс-валидацию и подбор гиперпараметров.

**Правила**
* Код пишите вместо `# <ВАШ КОД>`, переменные с `None` заполните своими значениями.
* Ячейки с пометкой `# Проверка (не изменяйте)` должны выполняться без ошибок.
* Гиперпараметры, признаки и порог подбирайте **только на обучающей выборке** (через кросс-валидацию). Тестовая выборка используется один раз, в самом конце, для итоговой оценки.
* На вопросы отвечайте текстом в ячейках «**Ответ:**», своими словами, 2–4 предложения.

**Баллы:** часть 1 — 2, часть 2 — 2.5, часть 3 — 2.5, часть 4 — 2, часть 5 — 1. Всего 10.

In [ ]:
!pip install -q optuna

In [ ]:
import hashlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes, fetch_openml
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix,
                             precision_score, r2_score, recall_score, roc_auc_score)
from sklearn.model_selection import (GridSearchCV, KFold, StratifiedKFold, cross_val_predict,
                                     cross_val_score, train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# Часть 1. Регуляризация (2 балла)

Датасет `diabetes` встроен в sklearn: 442 пациента, 10 признаков (возраст, пол, индекс массы тела `bmi`, давление `bp` и шесть показателей анализа крови `s1`–`s6`). Целевая переменная — количественная мера прогрессирования диабета через год после обследования.

Признаки в датасете уже отцентрированы и отнормированы.

In [ ]:
X_diab, y_diab = load_diabetes(return_X_y=True, as_frame=True)
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(X_diab, y_diab, test_size=0.25,
                                                  random_state=RANDOM_STATE)
X_tr_d.shape, X_te_d.shape

## 1.1 Переобучение полиномиальной модели

Постройте модель `make_pipeline(PolynomialFeatures(3, include_bias=False), StandardScaler(), LinearRegression())`.
Посчитайте $R^2$ на обучающей и тестовой выборках.

In [ ]:
poly_lr = None  # <ВАШ КОД>

r2_train_lr = None  # <ВАШ КОД>
r2_test_lr = None   # <ВАШ КОД>
print(f"R2 train: {r2_train_lr:.3f}, R2 test: {r2_test_lr:.3f}")

In [ ]:
# Проверка (не изменяйте)
assert poly_lr[0].degree == 3, "Нужна степень полинома 3"
assert r2_train_lr > 0.8 and r2_test_lr < 0, "Модель должна переобучиться: высокий R2 на train и отрицательный на test"
print("OK")

## 1.2 Ridge и Lasso с подбором alpha по кросс-валидации

Замените `LinearRegression` в том же пайплайне на `Ridge`, а затем на `Lasso(max_iter=20000)`.
Для каждой модели подберите `alpha` из сетки `alphas` по среднему $R^2$ на 5-кратной кросс-валидации **на обучающей выборке**.

Используйте `cross_val_score(model, X, y, cv=cv, scoring="r2")` — функция сама делит данные на фолды, обучает модель на каждом разбиении и возвращает массив оценок.

Постройте график «среднее $R^2$ на CV в зависимости от alpha» для обеих моделей (ось X — логарифмическая).

In [ ]:
alphas = np.logspace(-1, 3, 13)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

ridge_cv_scores = []  # средний R2 на CV для каждого alpha
lasso_cv_scores = []

# <ВАШ КОД>

best_alpha_ridge = None  # <ВАШ КОД>
best_alpha_lasso = None  # <ВАШ КОД>

# график
# <ВАШ КОД>

Обучите Ridge и Lasso с лучшими `alpha` на всей обучающей выборке и посчитайте $R^2$ на тесте.
Для Lasso посчитайте, сколько коэффициентов из 285 осталось ненулевыми.

In [ ]:
ridge_best = None  # <ВАШ КОД>
lasso_best = None  # <ВАШ КОД>

r2_test_ridge = None  # <ВАШ КОД>
r2_test_lasso = None  # <ВАШ КОД>
n_nonzero_lasso = None  # <ВАШ КОД>

print(f"Ridge (alpha={best_alpha_ridge:.2f}): R2 test = {r2_test_ridge:.3f}")
print(f"Lasso (alpha={best_alpha_lasso:.2f}): R2 test = {r2_test_lasso:.3f}, ненулевых коэффициентов: {n_nonzero_lasso}")

In [ ]:
# Проверка (не изменяйте)
assert len(ridge_cv_scores) == len(alphas) and len(lasso_cv_scores) == len(alphas)
assert r2_test_ridge > 0.35, "Ridge с подобранным alpha должен заметно превзойти LinearRegression"
assert r2_test_lasso > 0.45, "Lasso с подобранным alpha должен давать R2 > 0.45"
assert 0 < n_nonzero_lasso < 285
print("OK")

## 1.3 Путь регуляризации Lasso

На **исходных 10 признаках** (без полиномов) обучите `Lasso` для каждого `alpha` из `path_alphas` и сохраните коэффициенты в массив `coefs` размера `(40, 10)`.
Постройте график: по оси X — alpha (логарифмическая шкала), по оси Y — коэффициенты, по одной линии на признак, с легендой.

In [ ]:
path_alphas = np.logspace(-1, 2, 40)
scaler_d = StandardScaler().fit(X_tr_d)
X_tr_d_scaled = scaler_d.transform(X_tr_d)

coefs = None  # <ВАШ КОД>

# график
# <ВАШ КОД>

Какие **два** признака Lasso обнуляет последними? Запишите их названия в `last_two` (порядок не важен).

In [ ]:
last_two = []  # <ВАШ КОД>, например ["age", "sex"]

In [ ]:
# Проверка (не изменяйте)
assert coefs.shape == (40, 10)
assert hashlib.md5(",".join(sorted(last_two)).encode()).hexdigest() == "13f104a52a26582f52733f857055217f", "Посмотрите на график ещё раз"
print("OK")

**Вопрос 1.** Почему LinearRegression на полиномах 3-й степени дала отрицательный $R^2$ на тесте, а Ridge и Lasso — нет? Чем отличается то, как Ridge и Lasso «борются» с лишними признаками?

**Ответ:**

# Часть 2. Предсказание отказов оборудования (2.5 балла)

Датасет [AI4I 2020 Predictive Maintenance](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset) — синтетические, но реалистичные данные о работе фрезерного станка: 10 000 записей.

| Столбец | Описание |
|---|---|
| `type` | качество изделия: L (низкое), M (среднее), H (высокое) |
| `air_temp`, `process_temp` | температура воздуха и процесса, К |
| `rpm` | скорость вращения шпинделя, об/мин |
| `torque` | крутящий момент, Н·м |
| `tool_wear` | износ инструмента, мин |
| **`failure`** | **целевая переменная: 1 — произошёл отказ** |
| `TWF`, `HDF`, `PWF`, `OSF`, `RNF` | тип отказа: износ инструмента, перегрев, мощность, перегрузка, случайный |
| `UDI`, `Product ID` | идентификаторы |

Задача: по показаниям датчиков предсказать отказ.

In [ ]:
df = fetch_openml(data_id=42890, as_frame=True, parser="auto").frame
df = df.rename(columns={
    "Type": "type",
    "Air temperature [K]": "air_temp",
    "Process temperature [K]": "process_temp",
    "Rotational speed [rpm]": "rpm",
    "Torque [Nm]": "torque",
    "Tool wear [min]": "tool_wear",
    "Machine failure": "failure",
})
df.head()

## 2.1 Подготовка признаков

Соберите матрицу признаков `X` и целевую переменную `y`:
* удалите идентификаторы `UDI` и `Product ID`;
* удалите целевую переменную и **столбцы с типами отказов** — они известны только после того, как отказ случился, и в момент прогноза их нет;
* закодируйте `type` через one-hot (`pd.get_dummies(..., columns=["type"], dtype=float)`).

Посчитайте долю отказов в данных.

In [ ]:
FAILURE_TYPES = ["TWF", "HDF", "PWF", "OSF", "RNF"]

X = None  # <ВАШ КОД>
y = None  # <ВАШ КОД>

failure_rate = None  # <ВАШ КОД>
print(f"Доля отказов: {failure_rate:.2%}")
X.head()

In [ ]:
# Проверка (не изменяйте)
assert X.shape == (10000, 8), f"Ожидалось 8 признаков, получилось {X.shape[1]}"
assert not set(FAILURE_TYPES + ["failure", "UDI", "Product ID"]) & set(X.columns), "В признаках остались лишние столбцы"
assert abs(failure_rate - 0.0339) < 1e-4
print("OK")

Разбиение на обучающую и тестовую выборки (не изменяйте). Параметр `stratify=y` сохраняет долю отказов одинаковой в обеих частях — при редком классе это важно.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y,
                                                    random_state=RANDOM_STATE)
y_train.mean(), y_test.mean()

## 2.2 Базовые модели и метрики

Обучите на `X_train`:
* `DummyClassifier()` — всегда предсказывает самый частый класс;
* `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))`.

Для каждой модели посчитайте на `X_test` accuracy, precision, recall, ROC-AUC и PR-AUC (`average_precision_score`; для ROC-AUC и PR-AUC нужны вероятности из `predict_proba`, а не метки классов).
Сложите результаты в таблицу `metrics` (строки — модели `"dummy"` и `"logreg"`, столбцы — `"accuracy"`, `"precision"`, `"recall"`, `"roc_auc"`, `"pr_auc"`). Выведите матрицу ошибок логистической регрессии.

In [ ]:
dummy = None   # <ВАШ КОД>
logreg = None  # <ВАШ КОД>

metrics = None  # <ВАШ КОД>
metrics

In [ ]:
# Проверка (не изменяйте)
assert set(metrics.index) == {"dummy", "logreg"}
assert set(metrics.columns) >= {"accuracy", "precision", "recall", "roc_auc", "pr_auc"}
assert metrics.loc["dummy", "accuracy"] > 0.96 and metrics.loc["dummy", "recall"] == 0
assert metrics.loc["logreg", "roc_auc"] > 0.85
assert metrics.loc["logreg", "pr_auc"] < metrics.loc["logreg", "roc_auc"]
print("OK")

**Вопрос 2.** У DummyClassifier accuracy около 0.97. Почему эта модель бесполезна для задачи, и почему PR-AUC здесь информативнее, чем accuracy и ROC-AUC?

**Ответ:**

## 2.3 Утечка данных

Проведите эксперимент: обучите ту же логистическую регрессию, добавив к признакам столбцы с типами отказов `FAILURE_TYPES` (используйте те же индексы train/test). Посчитайте ROC-AUC на тесте.

In [ ]:
roc_auc_leak = None  # <ВАШ КОД>
print(f"ROC-AUC с типами отказов: {roc_auc_leak:.3f}")

In [ ]:
# Проверка (не изменяйте)
assert roc_auc_leak > 0.95
print("OK")

**Вопрос 3.** Почему качество так выросло? Можно ли использовать такую модель на реальном станке? Приведите пример похожей утечки из вашей предметной области.

**Ответ:**

# Часть 3. Деревья, ансамбли и кросс-валидация (2.5 балла)

## 3.1 Сравнение моделей

Сравните четыре модели по PR-AUC на 5-кратной **стратифицированной** кросс-валидации на обучающей выборке (`scoring="average_precision"`):
* логистическая регрессия (как в 2.2);
* `DecisionTreeClassifier(random_state=RANDOM_STATE)`;
* `RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)`;
* `XGBClassifier(random_state=RANDOM_STATE, n_jobs=-1)`.

Сохраните результаты в таблицу `cv_results`: строки — `"logreg"`, `"tree"`, `"forest"`, `"xgb"`, столбцы — `"mean"` и `"std"`.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    # <ВАШ КОД>
}

cv_results = None  # <ВАШ КОД>
cv_results

In [ ]:
# Проверка (не изменяйте)
assert set(cv_results.index) == {"logreg", "tree", "forest", "xgb"}
assert cv_results.loc[["forest", "xgb"], "mean"].min() > cv_results.loc[["logreg", "tree"], "mean"].max() + 0.1, \
    "Ансамбли должны заметно обгонять одиночное дерево и логистическую регрессию"
print("OK")

## 3.2 Важность признаков

Обучите случайный лес на всей обучающей выборке и постройте столбчатую диаграмму `feature_importances_`, отсортированную по убыванию.

In [ ]:
# <ВАШ КОД>

## 3.3 Признаки из физики процесса

Отказы в этом датасете вызваны физическими причинами: перегрев (мала разница температур при низкой скорости), отказ по мощности (мощность вне допустимого диапазона), перегрузка (износ × момент слишком велик).
Напишите функцию `add_physics_features`, которая добавляет к таблице три признака:
* `power` — механическая мощность, Вт: $P = M \cdot \omega$, где $\omega = 2\pi \cdot \text{rpm} / 60$;
* `temp_diff` — разность `process_temp - air_temp`;
* `strain` — произведение `tool_wear * torque`.

Сравните PR-AUC на CV для логистической регрессии и XGBoost без новых признаков и с ними.

In [ ]:
def add_physics_features(data):
    data = data.copy()
    # <ВАШ КОД>
    return data


X_train_fe = add_physics_features(X_train)
X_test_fe = add_physics_features(X_test)

fe_results = None  # таблица: строки "logreg", "xgb"; столбцы "base", "physics" (средний PR-AUC на CV)
# <ВАШ КОД>
fe_results

In [ ]:
# Проверка (не изменяйте)
assert {"power", "temp_diff", "strain"} <= set(X_train_fe.columns)
assert abs(X_train_fe["power"].median() - 6100) < 300, "Проверьте формулу мощности"
assert fe_results.loc["xgb", "physics"] > fe_results.loc["xgb", "base"] + 0.03
print("OK")

**Вопрос 4.** XGBoost заметно выиграл от новых признаков, а логистическая регрессия — почти нет. Почему? Подсказка: отказ по мощности случается, когда мощность **либо слишком мала, либо слишком велика**.

**Ответ:**

# Часть 4. Подбор гиперпараметров и выбор порога (2 балла)

Дальше работаем с XGBoost на признаках `X_train_fe`.

## 4.1 Optuna

Напишите функцию `objective(trial)`, которая предлагает гиперпараметры XGBoost и возвращает средний PR-AUC на кросс-валидации `cv` по `X_train_fe`. Подбирайте как минимум `n_estimators` (100–600), `max_depth` (2–8), `learning_rate` (0.01–0.3, логарифмическая шкала), `subsample` (0.6–1.0), `min_child_weight` (1–10).
Запустите 30 испытаний.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    # <ВАШ КОД>
    pass


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=30, show_progress_bar=True)
print("Лучший PR-AUC на CV:", study.best_value)
print("Лучшие параметры:", study.best_params)

In [ ]:
# Проверка (не изменяйте)
assert len(study.trials) >= 30
assert study.best_value > 0.85
print("OK")

## 4.2 Сравнение с GridSearchCV

Подберите те же гиперпараметры через `GridSearchCV` с сопоставимым бюджетом — **не больше 30 комбинаций** в сетке. Сравните лучший результат на CV и время работы с Optuna.

In [ ]:
param_grid = {
    # <ВАШ КОД>
}

grid = None  # <ВАШ КОД>

print("Комбинаций в сетке:", len(grid.cv_results_["params"]))
print("Лучший PR-AUC на CV:", grid.best_score_)
print("Лучшие параметры:", grid.best_params_)

In [ ]:
# Проверка (не изменяйте)
assert len(grid.cv_results_["params"]) <= 30
assert grid.scoring == "average_precision"
print("OK")

## 4.3 Выбор порога по стоимости ошибок

Модель выдаёт вероятность отказа, а решение «останавливать станок на обслуживание или нет» принимается по порогу. Пусть **пропущенный отказ стоит 20 условных единиц, а ложная тревога — 1**.

1. Возьмите XGBoost с лучшими параметрами Optuna и получите **out-of-fold** вероятности на обучающей выборке: `cross_val_predict(model, X_train_fe, y_train, cv=cv, method="predict_proba")`. Каждая вероятность в них предсказана моделью, которая этот объект не видела.
2. Подберите по ним порог из `thresholds`, минимизирующий суммарную стоимость ошибок.
3. Обучите модель на всей обучающей выборке и **один раз** оцените её на тесте: PR-AUC и стоимость ошибок при подобранном пороге и при пороге 0.5.

In [ ]:
COST_FN, COST_FP = 20, 1
thresholds = np.linspace(0.005, 0.995, 199)


def total_cost(y_true, proba, threshold):
    # <ВАШ КОД>
    pass


best_threshold = None  # <ВАШ КОД>

final_model = None  # <ВАШ КОД>
test_proba = None   # <ВАШ КОД>

test_pr_auc = average_precision_score(y_test, test_proba)
cost_best = total_cost(y_test, test_proba, best_threshold)
cost_05 = total_cost(y_test, test_proba, 0.5)
cost_never = total_cost(y_test, test_proba, 1.01)  # никогда не останавливать станок

print(f"PR-AUC на тесте: {test_pr_auc:.3f}")
print(f"Порог {best_threshold:.3f}: стоимость {cost_best}")
print(f"Порог 0.5: стоимость {cost_05}")
print(f"Без модели: стоимость {cost_never}")

In [ ]:
# Проверка (не изменяйте)
assert total_cost(np.array([1, 0, 1, 0]), np.array([0.9, 0.9, 0.1, 0.1]), 0.5) == COST_FN + COST_FP
assert cost_never == COST_FN * y_test.sum()
assert best_threshold < 0.5, "При дорогих пропусках порог должен быть ниже 0.5"
assert cost_best < cost_05
assert test_pr_auc > 0.8
print("OK")

# Часть 5. Выводы (1 балл)

**Вопрос 5.** Представьте, что модель внедряют для мониторинга реальных станков. Ответьте на вопросы:
* Какую модель и какой порог вы бы выбрали и почему?
* Почему порог подбирался по out-of-fold предсказаниям на train, а не по тесту?
* Что изменится, если пропуск отказа будет стоить не 20, а 200? Что ещё, кроме стоимости ошибок, нужно учесть перед внедрением?

**Ответ:**